## 11.3 GRU - 前向传播案例（完整流程 + 维度计算）

#### 1. 这一节的目标

##### 1.1 学完这一节后，应该能做到：

- 看懂一个 GRU 的输入张量 shape
- 知道每个时间步内部发生了什么
- 能手动推导各个门和状态的维度
- 能理解整个序列输出的 shape                                                                                                                                                                                

#### 2. 案例设定

##### 2.1 我们先设定一个固定案例

为了让整个过程清楚，我们使用下面这组参数：

- batch size = 2
- 序列长度 $T = 3$
- 输入维度 $d_x = 4$
- 隐藏维度 $d_h = 5$

##### 2.2 输入张量的形状

假设我们使用 PyTorch 中常见的：

```python
batch_first=True
```

那么输入张量为：

$X \in \mathbb{R}^{2 \times 3 \times 4}$

它表示：

- `2`：batch 中有 2 条样本
- `3`：每条样本有 3 个时间步
- `4`：每个时间步有 4 个输入特征

##### 2.3 这个输入张量怎么理解

可以把它想象成：

- 第 1 条样本：长度为 3 的序列
- 第 2 条样本：长度为 3 的序列

并且每个时间步输入都是一个 4 维向量。

所以整体可以理解为：

$X = [x_1,\ x_2,\ x_3]$

但这里的每个 $x_t$ 都不是单个样本的向量，而是：

$x_t \in \mathbb{R}^{2 \times 4}$

也就是 batch 中所有样本在第 $t$ 个时间步的输入。

#### 3. 先建立整体流程图

##### 3.1 GRU 的整体时间展开

对于序列长度 $T = 3$，GRU 的时间展开过程就是：

$(x_1,\ h_0) \rightarrow h_1$

$(x_2,\ h_1) \rightarrow h_2$

$(x_3,\ h_2) \rightarrow h_3$

这就是整个序列前向传播的主链路。

##### 3.2 关键理解一句话

整个序列 = 单时间步 GRU 计算，在时间维度上重复 3 次。

而且每一次使用的参数都相同，这就是循环网络的“参数共享”。


#### 4. 先看初始状态

##### 4.1 初始隐藏状态是什么

在第一个时间步之前，我们需要一个初始隐藏状态：

$h_0$

在实际中，通常默认初始化为全 0 张量。

##### 4.2 初始隐藏状态的维度

因为：

- batch size = 2
- hidden size = 5

所以：

$h_0 \in \mathbb{R}^{2 \times 5}$

你可以把它理解为：

- batch 中有 2 条样本
- 每条样本当前都有一个 5 维隐藏状态

#### 5. 时间步 1：完整前向传播

##### 5.1 当前输入是什么

在第一个时间步：

$x_1 \in \mathbb{R}^{2 \times 4}$

初始隐藏状态：

$h_0 \in \mathbb{R}^{2 \times 5}$

##### 5.2 计算更新门 $z_1$

公式为：

$z_1 = \sigma(W_z x_1 + U_z h_0 + b_z)$

在 batch 形式下，更准确地理解为：

- 输入部分输出 shape：$(2,\ 5)$
- 隐藏状态部分输出 shape：$(2,\ 5)$
- 相加后仍然是：$(2,\ 5)$

所以：

$z_1 \in \mathbb{R}^{2 \times 5}$

##### 5.3 计算重置门 $r_1$

公式为：

$r_1 = \sigma(W_r x_1 + U_r h_0 + b_r)$

同理：

$r_1 \in \mathbb{R}^{2 \times 5}$

##### 5.4 计算候选隐藏状态 $\tilde{h}_1$

公式为：

$\tilde{h}_1 = \tanh(W_h x_1 + U_h(r_1 \odot h_0) + b_h)$

先看中间部分：

$r_1 \odot h_0$

由于：

- $r_1 \in \mathbb{R}^{2 \times 5}$
- $h_0 \in \mathbb{R}^{2 \times 5}$

所以逐元素相乘后：

$r_1 \odot h_0 \in \mathbb{R}^{2 \times 5}$

继续线性变换后，最终得到：

$\tilde{h}_1 \in \mathbb{R}^{2 \times 5}$

##### 5.5 计算当前隐藏状态 $h_1$

公式为：

$h_1 = z_1 \odot h_0 + (1-z_1)\odot \tilde{h}_1$

因为三部分维度都是：

$(2,\ 5)$

所以最终：

$h_1 \in \mathbb{R}^{2 \times 5}$

##### 5.6 时间步 1 的结果总结

到这里，第一个时间步结束后，我们得到：

- $z_1 \in \mathbb{R}^{2 \times 5}$
- $r_1 \in \mathbb{R}^{2 \times 5}$
- $\tilde{h}_1 \in \mathbb{R}^{2 \times 5}$
- $h_1 \in \mathbb{R}^{2 \times 5}$

其中 $h_1$ 会被传给下一时间步继续使用。

#### 6. 时间步 2：重复同样流程

##### 6.1 当前输入和上一状态

在第二个时间步：

$x_2 \in \mathbb{R}^{2 \times 4}$

上一时刻隐藏状态：

$h_1 \in \mathbb{R}^{2 \times 5}$

##### 6.2 更新门

$z_2 = \sigma(W_z x_2 + U_z h_1 + b_z)$

所以：

$z_2 \in \mathbb{R}^{2 \times 5}$

##### 6.3 重置门

$r_2 = \sigma(W_r x_2 + U_r h_1 + b_r)$

所以：

$r_2 \in \mathbb{R}^{2 \times 5}$

##### 6.4 候选隐藏状态

$\tilde{h}_2 = \tanh(W_h x_2 + U_h(r_2 \odot h_1) + b_h)$

所以：

$\tilde{h}_2 \in \mathbb{R}^{2 \times 5}$

##### 6.5 当前隐藏状态

$h_2 = z_2 \odot h_1 + (1-z_2)\odot \tilde{h}_2$

所以：

$h_2 \in \mathbb{R}^{2 \times 5}$

##### 6.6 这一时间步的本质

你会发现，第 2 步和第 1 步没有任何新公式。

变化的只是：

- 输入从 $x_1$ 变成了 $x_2$
- 旧状态从 $h_0$ 变成了 $h_1$

这就再次说明：

多时间步并没有新的前向传播原理，只是在重复单步计算。

#### 7. 时间步 3：继续重复

##### 7.1 当前输入和上一状态

第三个时间步：

$x_3 \in \mathbb{R}^{2 \times 4}$

上一隐藏状态：

$h_2 \in \mathbb{R}^{2 \times 5}$

##### 7.2 各个门和状态

同样会得到：

- $z_3 \in \mathbb{R}^{2 \times 5}$
- $r_3 \in \mathbb{R}^{2 \times 5}$
- $\tilde{h}_3 \in \mathbb{R}^{2 \times 5}$
- $h_3 \in \mathbb{R}^{2 \times 5}$

##### 7.3 到这里整个序列结束

因为序列长度就是 3，所以时间步 3 算完以后，整个 GRU 前向传播就结束了。

#### 8. 把所有时间步输出堆叠起来

##### 8.1 每个时间步都有一个输出

GRU 和普通 RNN 一样，每个时间步都会输出一个隐藏状态：

$h_1,\ h_2,\ h_3$

##### 8.2 把它们沿时间维堆起来

如果使用 `batch_first=True`，那么最终整个输出张量为：

$H \in \mathbb{R}^{2 \times 3 \times 5}$

也就是：

```python
output.shape = (2, 3, 5)
```

含义是：

- `2`：batch size
- `3`：序列长度
- `5`：隐藏维度

##### 8.3 最后一个隐藏状态

除了整个序列输出，还会单独保留最后一个隐藏状态：

$h_3 \in \mathbb{R}^{2 \times 5}$

这个量在很多任务中都很重要。

#### 9. 用表格总结整个案例的维度变化

| 阶段 | 张量 | Shape |
|---|---|---|
| 输入整体 | $X$ | $(2,\ 3,\ 4)$ |
| 单步输入 | $x_t$ | $(2,\ 4)$ |
| 初始隐藏状态 | $h_0$ | $(2,\ 5)$ |
| 更新门 | $z_t$ | $(2,\ 5)$ |
| 重置门 | $r_t$ | $(2,\ 5)$ |
| 候选隐藏状态 | $\tilde{h}_t$ | $(2,\ 5)$ |
| 当前隐藏状态 | $h_t$ | $(2,\ 5)$ |
| 所有时间步输出 | $H$ | $(2,\ 3,\ 5)$ |
| 最终隐藏状态 | $h_3$ | $(2,\ 5)$ |